# Yelp reviews analyzed with different sentiment analysis methods and models

In [59]:
# Import the necessary libraries/packages
# Data processing
import pandas as pd
import numpy as np

# Import TextBlob
from textblob import TextBlob
from textblob import Blobber
from textblob.sentiments import NaiveBayesAnalyzer

# Import VADER sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Import flair pre-trained sentiment model
from flair.models import TextClassifier
classifier = TextClassifier.load('en-sentiment')

# Import flair Sentence to process input text
from flair.data import Sentence

# Import accuracy_score to check performance
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Set a wider colwith
pd.set_option('display.max_colwidth', 1000)

In [60]:
yelp = pd.read_csv("datasets/train.csv", header = None)

In [61]:
yelp.columns=["Label","Review"]

For the sentiment column labels are as follow : 1 is a generally negative review, 2 is a generally positive review. Cf Readme.txt

In [62]:
yelp.head(10)

,Label,Review
0,1,"Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the phone. It usually takes 2 hours of repeated calling to get an answer. Who has time for that or wants to deal with it? I have run into this problem with many other doctors and I just don't get it. You have office workers, you have patients with medical needs, why isn't anyone answering the phone? It's incomprehensible and not work the aggravation. It's with regret that I feel that I have to give Dr. Goldberg 2 stars."
1,2,"Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years and is really all about the big picture. It is because of him, not my now former gyn Dr. Markoff, that I found out I have fibroids. He explores all options with you and is very patient and understanding. He doesn't judge and asks all the right questions. Very thorough and wants to be kept in the loop on every aspect of your medical health and your life."
2,1,"I don't know what Dr. Goldberg was like before moving to Arizona, but let me tell you, STAY AWAY from this doctor and this office. I was going to Dr. Johnson before he left and Goldberg took over when Johnson left. He is not a caring doctor. He is only interested in the co-pay and having you come in for medication refills every month. He will not give refills and could less about patients's financial situations. Trying to get your 90 days mail away pharmacy prescriptions through this guy is a joke. And to make matters even worse, his office staff is incompetent. 90% of the time when you call the office, they'll put you through to a voice mail, that NO ONE ever answers or returns your call. Both my adult children and husband have decided to leave this practice after experiencing such frustration. The entire office has an attitude like they are doing you a favor. Give me a break! Stay away from this doc and the practice. You deserve better and they will not be there when you really ..."
3,1,"I'm writing this review to give you a heads up before you see this Doctor. The office staff and administration are very unprofessional. I left a message with multiple people regarding my bill, and no one ever called me back. I had to hound them to get an answer about my bill. \n\nSecond, and most important, make sure your insurance is going to cover Dr. Goldberg's visits and blood work. He recommended to me that I get a physical, and he knew I was a student because I told him. I got the physical done. Later, I found out my health insurance doesn't pay for preventative visits. I received an $800.00 bill for the blood work. I can't pay for my bill because I'm a student and don't have any cash flow at this current time. I can't believe the Doctor wouldn't give me a heads up to make sure my insurance would cover work that wasn't necessary and was strictly preventative. The office can't do anything to help me cover the bill. In addition, the office staff said the onus is on me to make s..."
4,2,"All the food is great here. But the best thing they have is their wings. Their wings are simply fantastic!! The \""Wet Cajun\"" are by the best & most popular. I also like the seasoned salt wings. Wing Night is Monday & Wednesday night, $0.75 whole wings!\n\nThe dining area is nice. Very family friendly! The bar is very nice is well. This place is truly a Yinzer's dream!! \""Pittsburgh Dad\"" would love this place n'at!!"
5,1,"Wing sauce is like water. Pretty much a lot of butter and some hot sauce (franks red hot maybe). The whole wings are good size and crispy, but for $1 a wing the sauce could be better. The hot and extra hot are about the same flavor/heat. The fish sandwich is good and is a large portion, sides are decent."
6,1,"Owning a driving range inside the city limits is like a license to print money. I 

In [63]:
yelp.info()

<class 'pandas.DataFrame'>
RangeIndex: 560000 entries, 0 to 559999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   Label   560000 non-null  int64
 1   Review  560000 non-null  str  
dtypes: int64(1), str(1)
memory usage: 8.5 MB


In [64]:
yelp["Label"].value_counts()

Label
1    280000
2    280000
Name: count, dtype: int64

Dataset is balanced between positive and negative review, no missing values

In [65]:
from sklearn.model_selection import train_test_split

yelp_sample, _ = train_test_split(
    yelp,
    train_size=10000,
    stratify=yelp["Label"],
    random_state=42
)

print(yelp_sample["Label"].value_counts())


Label
1    5000
2    5000
Name: count, dtype: int64


In [66]:
yelp_sample

,Label,Review
423072,1,"Maybe it was the build up, the long drive, whatever - but I expected more. \n\nBakery is either sweets or meat during the week - not including a tasty scallion bun I scarfed in record time - but the couple of cool-sounding vegetarian stuffs were unavailable. K.\n\nBuffet-wise, there was a really, really unappetizing looking egg dish - scrambled with tomato. Rest was beef/chicken/seafood. K.\n\nThere was an entire half-fridge section of interesting things done with soy - crepe, marinated, fried, baked, gluten products as well. A tad more comprehensive than other, closer stores. Pretty rad sake selection. \n\nStill. If the bakery/restaurant was more veg friendly, it'd be worth it, the 20 minute drive. As it stands, not so much. A clean floor does not a worthy adventure make."
507689,1,"I like the idea of this place ... a walk-up cafe with mix-and-match options, made to order. Nice. But, you know, the food actually has to taste good if I'm ever to come back. And, sorry, I won't.\n\nI had the Budha Bowl. Yes, they inexplicably do not spell Buddha correctly -- what, are they actually trying to brand the word \""Budha\""? A bowl of noodles with an obnoxious amount of onions and peppers and few other veggies. Bathed in a boring \""hosin-ginger sauce\"". I put the sauce name in quotes because I didn't taste ginger, or any other distinguishing flavor, for that matter ... I'm guessing it is not made on site. A quite-small smattering of mung bean sprouts and a lime on top. Yawn."
262772,1,"I'll start off by saying the service was great.\n\nas for the food, got the 9oz medium rare wagyu ribeye with bone marrow added, side of polenta, side of mac n cheese, and a side of creamed spinach.\n\nthe ribeye was cooked right, but a lil salty. the marrow ruined the steak, so much grease it made me feel sick. it covered the bottom of the plate and all the sides that they put on the plate soaked it up, see pic here:\n\nhttp://www.yelp.com/biz_photos/J4CATH00YZrq8Bne2S4_cw?select=9g_-bc64OcVN7ZS_qQAsmQ\n\nif you get marrow, I recommend gettin it on the side.\n\npolenta was okay, mac n cheese was good but salty, the creamed spinach was not as good as other places. disappointing meal for droppin $150 a head."
537954,1,"Eh... Was going to order waffles but they had no ice cream, so I went with the avocado drink. Ummmm... No bueno. Too much ice, no flavor, and I don't think the avocado was ripe enough. I make avocado drinks and it should turn somewhat green. My drink was a pale green yellow. The lady was nice so I gave her a star and then another star for the drink. Absolute fail. Sorry guys... Just speaking the truth. Make the avacado drink more milky less icy. Or just don't call it a smoothie cause it's far from that."
30428,2,"This club is top notch in all respects. not only is it gorgeous and way out of my league, but it is a lot of fun. I truly felt like Tiger (or his cousin Panther) off the tee because that thin air let that ball travel. The greens play slower than what i'm used to in CA and NV, so put some oomph into your puts. I have never seen so many streams, lakes, and waterfalls through 18 holes. \n\nMore than anything, the staff is very friendly and accommodating. I felt like a king - this isn't out of the ordinary for a place of this caliber. What is is the food. The food was excellent and priced similar to a cafe instead of a golf course. I ate the buffet breakfast and was wowed - this was the best breakfast I've eaten under $10 (tax and tip included!) anywhere. It even beat some more expensive breakfasts. The buffet lunch after our golf tourney was also excellent all around. Also, the wait staff was perfect. \n\nCome here if you want to live the country club life. After a day ..."
...,...,...
131876,1,"Los Dos was not up to par today. Green chile sauce had a strange lumpy consistency and the carne adovada was dry. The chicken in the chimichangas was tough, dry, and hard to swallow. The fire in the chile sauces was s

we'll rename label negative to 0 and positive to 1 for standard usage

In [68]:
yelp_sample['Label'] = yelp_sample['Label'].apply(lambda x: 1 if x == 2 else 0) 

In [69]:
yelp_sample

,Label,Review
423072,0,"Maybe it was the build up, the long drive, whatever - but I expected more. \n\nBakery is either sweets or meat during the week - not including a tasty scallion bun I scarfed in record time - but the couple of cool-sounding vegetarian stuffs were unavailable. K.\n\nBuffet-wise, there was a really, really unappetizing looking egg dish - scrambled with tomato. Rest was beef/chicken/seafood. K.\n\nThere was an entire half-fridge section of interesting things done with soy - crepe, marinated, fried, baked, gluten products as well. A tad more comprehensive than other, closer stores. Pretty rad sake selection. \n\nStill. If the bakery/restaurant was more veg friendly, it'd be worth it, the 20 minute drive. As it stands, not so much. A clean floor does not a worthy adventure make."
507689,0,"I like the idea of this place ... a walk-up cafe with mix-and-match options, made to order. Nice. But, you know, the food actually has to taste good if I'm ever to come back. And, sorry, I won't.\n\nI had the Budha Bowl. Yes, they inexplicably do not spell Buddha correctly -- what, are they actually trying to brand the word \""Budha\""? A bowl of noodles with an obnoxious amount of onions and peppers and few other veggies. Bathed in a boring \""hosin-ginger sauce\"". I put the sauce name in quotes because I didn't taste ginger, or any other distinguishing flavor, for that matter ... I'm guessing it is not made on site. A quite-small smattering of mung bean sprouts and a lime on top. Yawn."
262772,0,"I'll start off by saying the service was great.\n\nas for the food, got the 9oz medium rare wagyu ribeye with bone marrow added, side of polenta, side of mac n cheese, and a side of creamed spinach.\n\nthe ribeye was cooked right, but a lil salty. the marrow ruined the steak, so much grease it made me feel sick. it covered the bottom of the plate and all the sides that they put on the plate soaked it up, see pic here:\n\nhttp://www.yelp.com/biz_photos/J4CATH00YZrq8Bne2S4_cw?select=9g_-bc64OcVN7ZS_qQAsmQ\n\nif you get marrow, I recommend gettin it on the side.\n\npolenta was okay, mac n cheese was good but salty, the creamed spinach was not as good as other places. disappointing meal for droppin $150 a head."
537954,0,"Eh... Was going to order waffles but they had no ice cream, so I went with the avocado drink. Ummmm... No bueno. Too much ice, no flavor, and I don't think the avocado was ripe enough. I make avocado drinks and it should turn somewhat green. My drink was a pale green yellow. The lady was nice so I gave her a star and then another star for the drink. Absolute fail. Sorry guys... Just speaking the truth. Make the avacado drink more milky less icy. Or just don't call it a smoothie cause it's far from that."
30428,1,"This club is top notch in all respects. not only is it gorgeous and way out of my league, but it is a lot of fun. I truly felt like Tiger (or his cousin Panther) off the tee because that thin air let that ball travel. The greens play slower than what i'm used to in CA and NV, so put some oomph into your puts. I have never seen so many streams, lakes, and waterfalls through 18 holes. \n\nMore than anything, the staff is very friendly and accommodating. I felt like a king - this isn't out of the ordinary for a place of this caliber. What is is the food. The food was excellent and priced similar to a cafe instead of a golf course. I ate the buffet breakfast and was wowed - this was the best breakfast I've eaten under $10 (tax and tip included!) anywhere. It even beat some more expensive breakfasts. The buffet lunch after our golf tourney was also excellent all around. Also, the wait staff was perfect. \n\nCome here if you want to live the country club life. After a day ..."
...,...,...
131876,0,"Los Dos was not up to par today. Green chile sauce had a strange lumpy consistency and the carne adovada was dry. The chicken in the chimichangas was tough, dry, and hard to swallow. The fire in the chile sauces was s

## TextBlob Pattern Analyzer pipeline

In [70]:
yelp_sample["TextBlobPattern_scores"] = yelp_sample['Review'].apply(lambda s: TextBlob(s).sentiment.polarity)

In [71]:
# Predict sentiment label for each review
yelp_sample['pred_TextBlob'] = yelp_sample['TextBlobPattern_scores'].apply(lambda x: 1 if x >=0 else 0)
yelp_sample.head(10)

,Label,Review,TextBlobPattern_scores,pred_TextBlob
423072,0,"Maybe it was the build up, the long drive, whatever - but I expected more. \n\nBakery is either sweets or meat during the week - not including a tasty scallion bun I scarfed in record time - but the couple of cool-sounding vegetarian stuffs were unavailable. K.\n\nBuffet-wise, there was a really, really unappetizing looking egg dish - scrambled with tomato. Rest was beef/chicken/seafood. K.\n\nThere was an entire half-fridge section of interesting things done with soy - crepe, marinated, fried, baked, gluten products as well. A tad more comprehensive than other, closer stores. Pretty rad sake selection. \n\nStill. If the bakery/restaurant was more veg friendly, it'd be worth it, the 20 minute drive. As it stands, not so much. A clean floor does not a worthy adventure make.",0.146429,1
507689,0,"I like the idea of this place ... a walk-up cafe with mix-and-match options, made to order. Nice. But, you know, the food actually has to taste good if I'm ever to come back. And, sorry, I won't.\n\nI had the Budha Bowl. Yes, they inexplicably do not spell Buddha correctly -- what, are they actually trying to brand the word \""Budha\""? A bowl of noodles with an obnoxious amount of onions and peppers and few other veggies. Bathed in a boring \""hosin-ginger sauce\"". I put the sauce name in quotes because I didn't taste ginger, or any other distinguishing flavor, for that matter ... I'm guessing it is not made on site. A quite-small smattering of mung bean sprouts and a lime on top. Yawn.",0.012500,1
262772,0,"I'll start off by saying the service was great.\n\nas for the food, got the 9oz medium rare wagyu ribeye with bone marrow added, side of polenta, side of mac n cheese, and a side of creamed spinach.\n\nthe ribeye was cooked right, but a lil salty. the marrow ruined the steak, so much grease it made me feel sick. it covered the bottom of the plate and all the sides that they put on the plate soaked it up, see pic here:\n\nhttp://www.yelp.com/biz_photos/J4CATH00YZrq8Bne2S4_cw?select=9g_-bc64OcVN7ZS_qQAsmQ\n\nif you get marrow, I recommend gettin it on the side.\n\npolenta was okay, mac n cheese was good but salty, the creamed spinach was not as good as other places. disappointing meal for droppin $150 a head.",0.138492,1
537954,0,"Eh... Was going to order waffles but they had no ice cream, so I went with the avocado drink. Ummmm... No bueno. Too much ice, no flavor, and I don't think the avocado was ripe enough. I make avocado drinks and it should turn somewhat green. My drink was a pale green yellow. The lady was nice so I gave her a star and then another star for the drink. Absolute fail. Sorry guys... Just speaking the truth. Make the avacado drink more milky less icy. Or just don't call it a smoothie cause it's far from that.",-0.019762,0
30428,1,"This club is top notch in all respects. not only is it gorgeous and way out of my league, but it is a lot of fun. I truly felt like Tiger (or his cousin Panther) off the tee because that thin air let that ball travel. The greens play slower than what i'm used to in CA and NV, so put some oomph into your puts. I have never seen so many streams, lakes, and waterfalls through 18 holes. \n\nMore than anything, the staff is very friendly and accommodating. I felt like a king - this isn't out of the ordinary for a place of this caliber. What is is the food. The food was excellent and priced similar to a cafe instead of a golf course. I ate the buffet breakfast and was wowed - this was the best breakfast I've eaten under $10 (tax and tip included!) anywhere. It even beat some more expensive breakfasts. The buffet lunch after our golf tourney was also excellent all around. Also, the wait staff was perfect. \n\nCome here if you want to live the country club life. After a day ...",0.319381,1
302714,1,"I have a friend who is obsessed with Hunter S. Thompson. His obsession led to him typing only on a typewriter and deciding to take up ci

In [72]:
# Compare Actual and Predicted and get the Accuracy, Precision, Recall, and F1 score
actual_labels = yelp_sample['Label']
predicted_labels = yelp_sample['pred_TextBlob']

accuracy_TextBlob = accuracy_score(actual_labels,predicted_labels)
precision_TextBlob = precision_score(actual_labels, predicted_labels, average='macro')  # Use 'micro' or 'weighted' if more appropriate
recall_TextBlob = recall_score(actual_labels, predicted_labels, average='macro')
f1_TextBlob = f1_score(actual_labels, predicted_labels, average='macro')
report = classification_report(actual_labels, predicted_labels)

print(f'Accuracy:', accuracy_TextBlob)
print(f'Precision:', precision_TextBlob)
print(f'Recall:', recall_TextBlob)
print(f'F1:', f1_TextBlob)
print(f"Report : {report}")

Accuracy: 0.681
Precision: 0.7768492366547655
Recall: 0.6809999999999999
F1: 0.6507731444679467
Report :               precision    recall  f1-score   support

           0       0.94      0.39      0.55      5000
           1       0.61      0.98      0.75      5000

    accuracy                           0.68     10000
   macro avg       0.78      0.68      0.65     10000
weighted avg       0.78      0.68      0.65     10000



## TextBlob NaiveBayesAnalyzer pipeline

In [73]:
tb = Blobber(analyzer=NaiveBayesAnalyzer())

def predict_nb(texte):
    sentiment = tb(texte).sentiment
    return pd.Series({
        'naive_bayes_analysis': sentiment.classification,
        'nb_p_pos': sentiment.p_pos,   
        'nb_p_neg': sentiment.p_neg,    
        'pred_TextBlob_NaiveBayes': 1 if sentiment.classification == 'pos' else 0
    })

yelp_sample[['naive_bayes_analysis', 'nb_p_pos', 'nb_p_neg', 'pred_TextBlob_NaiveBayes']] = \
    yelp_sample['Review'].apply(predict_nb)

yelp_sample.head(10)

,Label,Review,TextBlobPattern_scores,pred_TextBlob,naive_bayes_analysis,nb_p_pos,nb_p_neg,pred_TextBlob_NaiveBayes
423072,0,"Maybe it was the build up, the long drive, whatever - but I expected more. \n\nBakery is either sweets or meat during the week - not including a tasty scallion bun I scarfed in record time - but the couple of cool-sounding vegetarian stuffs were unavailable. K.\n\nBuffet-wise, there was a really, really unappetizing looking egg dish - scrambled with tomato. Rest was beef/chicken/seafood. K.\n\nThere was an entire half-fridge section of interesting things done with soy - crepe, marinated, fried, baked, gluten products as well. A tad more comprehensive than other, closer stores. Pretty rad sake selection. \n\nStill. If the bakery/restaurant was more veg friendly, it'd be worth it, the 20 minute drive. As it stands, not so much. A clean floor does not a worthy adventure make.",0.146429,1,neg,0.163820,0.836180,0
507689,0,"I like the idea of this place ... a walk-up cafe with mix-and-match options, made to order. Nice. But, you know, the food actually has to taste good if I'm ever to come back. And, sorry, I won't.\n\nI had the Budha Bowl. Yes, they inexplicably do not spell Buddha correctly -- what, are they actually trying to brand the word \""Budha\""? A bowl of noodles with an obnoxious amount of onions and peppers and few other veggies. Bathed in a boring \""hosin-ginger sauce\"". I put the sauce name in quotes because I didn't taste ginger, or any other distinguishing flavor, for that matter ... I'm guessing it is not made on site. A quite-small smattering of mung bean sprouts and a lime on top. Yawn.",0.012500,1,neg,0.000355,0.999645,0
262772,0,"I'll start off by saying the service was great.\n\nas for the food, got the 9oz medium rare wagyu ribeye with bone marrow added, side of polenta, side of mac n cheese, and a side of creamed spinach.\n\nthe ribeye was cooked right, but a lil salty. the marrow ruined the steak, so much grease it made me feel sick. it covered the bottom of the plate and all the sides that they put on the plate soaked it up, see pic here:\n\nhttp://www.yelp.com/biz_photos/J4CATH00YZrq8Bne2S4_cw?select=9g_-bc64OcVN7ZS_qQAsmQ\n\nif you get marrow, I recommend gettin it on the side.\n\npolenta was okay, mac n cheese was good but salty, the creamed spinach was not as good as other places. disappointing meal for droppin $150 a head.",0.138492,1,pos,0.526412,0.473588,1
537954,0,"Eh... Was going to order waffles but they had no ice cream, so I went with the avocado drink. Ummmm... No bueno. Too much ice, no flavor, and I don't think the avocado was ripe enough. I make avocado drinks and it should turn somewhat green. My drink was a pale green yellow. The lady was nice so I gave her a star and then another star for the drink. Absolute fail. Sorry guys... Just speaking the truth. Make the avacado drink more milky less icy. Or just don't call it a smoothie cause it's far from that.",-0.019762,0,neg,0.286797,0.713203,0
30428,1,"This club is top notch in all respects. not only is it gorgeous and way out of my league, but it is a lot of fun. I truly felt like Tiger (or his cousin Panther) off the tee because that thin air let that ball travel. The greens play slower than what i'm used to in CA and NV, so put some oomph into your puts. I have never seen so many streams, lakes, and waterfalls through 18 holes. \n\nMore than anything, the staff is very friendly and accommodating. I felt like a king - this isn't out of the ordinary for a place of this caliber. What is is the food. The food was excellent and priced similar to a cafe instead of a golf course. I ate the buffet breakfast and was wowed - this was the best breakfast I've eaten under $10 (tax and tip included!) anywhere. It even beat some more expensive breakfasts. The buffet lunch after our golf tourney was also excellent all around. Also, the wait staff was perfect. \n\nCome here if you want to live the country club life. After a day ..

In [75]:
# Compare Actual and Predicted and get the Accuracy, Precision, Recall, and F1 score 
actual_labels = yelp_sample['Label']
predicted_labels = yelp_sample['pred_TextBlob_NaiveBayes']

accuracy_TextBlob_NB = accuracy_score(actual_labels,predicted_labels)
precision_TextBlob_NB = precision_score(actual_labels, predicted_labels, average='macro')  # Use 'micro' or 'weighted' if more appropriate
recall_TextBlob_NB = recall_score(actual_labels, predicted_labels, average='macro')
f1_TextBlob_NB = f1_score(actual_labels, predicted_labels, average='macro')
report = classification_report(actual_labels, predicted_labels)

print(f'Accuracy:', accuracy_TextBlob_NB)
print(f'Precision:', precision_TextBlob_NB)
print(f'Recall:', recall_TextBlob_NB)
print(f'F1:', f1_TextBlob_NB)
print(f"Report : {report}")

Accuracy: 0.6471
Precision: 0.6779913964936348
Recall: 0.6471
F1: 0.6310935590116658
Report :               precision    recall  f1-score   support

           0       0.75      0.44      0.55      5000
           1       0.60      0.86      0.71      5000

    accuracy                           0.65     10000
   macro avg       0.68      0.65      0.63     10000
weighted avg       0.68      0.65      0.63     10000



## VADER pipeline


In [76]:
# Get sentiment score for each review
vader_sentiment = SentimentIntensityAnalyzer()
yelp_sample['scores_VADER'] = yelp_sample['Review'].apply(lambda s: vader_sentiment.polarity_scores(s)['compound'])

# Predict sentiment label for each review
yelp_sample['pred_VADER'] = yelp_sample['scores_VADER'].apply(lambda x: 1 if x >=0 else 0)
yelp_sample.head(10)

,Label,Review,TextBlobPattern_scores,pred_TextBlob,naive_bayes_analysis,nb_p_pos,nb_p_neg,pred_TextBlob_NaiveBayes,scores_VADER,pred_VADER
423072,0,"Maybe it was the build up, the long drive, whatever - but I expected more. \n\nBakery is either sweets or meat during the week - not including a tasty scallion bun I scarfed in record time - but the couple of cool-sounding vegetarian stuffs were unavailable. K.\n\nBuffet-wise, there was a really, really unappetizing looking egg dish - scrambled with tomato. Rest was beef/chicken/seafood. K.\n\nThere was an entire half-fridge section of interesting things done with soy - crepe, marinated, fried, baked, gluten products as well. A tad more comprehensive than other, closer stores. Pretty rad sake selection. \n\nStill. If the bakery/restaurant was more veg friendly, it'd be worth it, the 20 minute drive. As it stands, not so much. A clean floor does not a worthy adventure make.",0.146429,1,neg,0.163820,0.836180,0,0.9756,1
507689,0,"I like the idea of this place ... a walk-up cafe with mix-and-match options, made to order. Nice. But, you know, the food actually has to taste good if I'm ever to come back. And, sorry, I won't.\n\nI had the Budha Bowl. Yes, they inexplicably do not spell Buddha correctly -- what, are they actually trying to brand the word \""Budha\""? A bowl of noodles with an obnoxious amount of onions and peppers and few other veggies. Bathed in a boring \""hosin-ginger sauce\"". I put the sauce name in quotes because I didn't taste ginger, or any other distinguishing flavor, for that matter ... I'm guessing it is not made on site. A quite-small smattering of mung bean sprouts and a lime on top. Yawn.",0.012500,1,neg,0.000355,0.999645,0,0.6124,1
262772,0,"I'll start off by saying the service was great.\n\nas for the food, got the 9oz medium rare wagyu ribeye with bone marrow added, side of polenta, side of mac n cheese, and a side of creamed spinach.\n\nthe ribeye was cooked right, but a lil salty. the marrow ruined the steak, so much grease it made me feel sick. it covered the bottom of the plate and all the sides that they put on the plate soaked it up, see pic here:\n\nhttp://www.yelp.com/biz_photos/J4CATH00YZrq8Bne2S4_cw?select=9g_-bc64OcVN7ZS_qQAsmQ\n\nif you get marrow, I recommend gettin it on the side.\n\npolenta was okay, mac n cheese was good but salty, the creamed spinach was not as good as other places. disappointing meal for droppin $150 a head.",0.138492,1,pos,0.526412,0.473588,1,-0.8205,0
537954,0,"Eh... Was going to order waffles but they had no ice cream, so I went with the avocado drink. Ummmm... No bueno. Too much ice, no flavor, and I don't think the avocado was ripe enough. I make avocado drinks and it should turn somewhat green. My drink was a pale green yellow. The lady was nice so I gave her a star and then another star for the drink. Absolute fail. Sorry guys... Just speaking the truth. Make the avacado drink more milky less icy. Or just don't call it a smoothie cause it's far from that.",-0.019762,0,neg,0.286797,0.713203,0,-0.7876,0
30428,1,"This club is top notch in all respects. not only is it gorgeous and way out of my league, but it is a lot of fun. I truly felt like Tiger (or his cousin Panther) off the tee because that thin air let that ball travel. The greens play slower than what i'm used to in CA and NV, so put some oomph into your puts. I have never seen so many streams, lakes, and waterfalls through 18 holes. \n\nMore than anything, the staff is very friendly and accommodating. I felt like a king - this isn't out of the ordinary for a place of this caliber. What is is the food. The food was excellent and priced similar to a cafe instead of a golf course. I ate the buffet breakfast and was wowed - this was the best breakfast I've eaten under $10 (tax and tip included!) anywhere. It even beat some more expensive breakfasts. The buffet lunch after our golf tourney was also excellent all around. Also, the wait staff was perfect. \n\nCome 

In [77]:
# Compare Actual and Predicted and get the Accuracy, Precision, Recall, and F1 score
actual_labels = yelp_sample['Label']
predicted_labels = yelp_sample['pred_VADER']

accuracy_VADER = accuracy_score(actual_labels,predicted_labels)
precision_VADER = precision_score(actual_labels, predicted_labels, average='macro')  # Use 'micro' or 'weighted' if more appropriate
recall_VADER = recall_score(actual_labels, predicted_labels, average='macro')
f1_VADER = f1_score(actual_labels, predicted_labels, average='macro')
report = classification_report(actual_labels, predicted_labels)

print(f'Accuracy:', accuracy_VADER)
print(f'Precision:', precision_VADER)
print(f'Recall:', recall_VADER)
print(f'F1:', f1_VADER)
print(f"Report : {report}")

Accuracy: 0.706
Precision: 0.7892141215561899
Recall: 0.706
F1: 0.6832131274411176
Report :               precision    recall  f1-score   support

           0       0.94      0.44      0.60      5000
           1       0.63      0.97      0.77      5000

    accuracy                           0.71     10000
   macro avg       0.79      0.71      0.68     10000
weighted avg       0.79      0.71      0.68     10000



## VADER pipeline

In [78]:
def score_flair(text):
  sentence = Sentence(text)
  classifier.predict(sentence)
  score = sentence.labels[0].score
  value = sentence.labels[0].value
  return score, value

# Batch flair

In [79]:
from flair.models import TextClassifier
from flair.data import Sentence


classifier = TextClassifier.load('en-sentiment')

def score_flair_batch(textes, batch_size=32):
    """Treat all samples as text — way faster than one by one."""
    sentences = [Sentence(str(t)) for t in textes]
    classifier.predict(sentences, mini_batch_size=batch_size)
    
    scores = []
    preds = []

    for sent in sentences:
        label = sent.labels[0]
        score = label.score
        # POSITIVE → 1, NEGATIVE → 0 (à adapter selon ton mapping)
        pred = 1 if label.value == 'POSITIVE' else 0
        scores.append(score)
        preds.append(pred)
    
    return scores, preds



In [80]:
scores, preds = score_flair_batch(yelp_sample['Review'].tolist(), batch_size=32)
yelp_sample['scores_flair'] = scores
yelp_sample['pred_flair'] = preds

In [81]:
# Compare Actual and Predicted and get the Accuracy, Precision, Recall, and F1 score
actual_labels = yelp_sample['Label']
predicted_labels = yelp_sample['pred_flair']

accuracy_flair = accuracy_score(actual_labels,predicted_labels)
precision_flair = precision_score(actual_labels, predicted_labels, average='macro')  # Use 'micro' or 'weighted' if more appropriate
recall_flair = recall_score(actual_labels, predicted_labels, average='macro')
f1_flair = f1_score(actual_labels, predicted_labels, average='macro')
report = classification_report(actual_labels, predicted_labels)

print(f'Accuracy:', accuracy_flair)
print(f'Precision:', precision_flair)
print(f'Recall:', recall_flair)
print(f'F1:', f1_flair)
print(f"Report : {report}")

Accuracy: 0.9361
Precision: 0.9375015625055804
Recall: 0.9361
F1: 0.9360487821091034
Report :               precision    recall  f1-score   support

           0       0.91      0.96      0.94      5000
           1       0.96      0.91      0.93      5000

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000



## Final Decision & Reflection

1. The Winner: 

The winner is flair by far if we solely base our analysis on metrics like F1_Score in particular, with a close to 0.94 Accuracy and F1_Score. 

Our dataset was complex, and a compilation of long reviews, with several sentences for most of them. It seems that simple methods which are not utilising neural networks, failed to capture the complexity of the language stucture (lots of words, order of words and context).


VADER ruleset improved the score a bit from a simple lexicon based prediction, but was still far from being as effective as flair.


2. The Trade-Off:

The trade-off for flair performance is obviously the execution time, being way longer than every other options (a few seconds vs 9 minutes almost with batch processing of 32).

Being a deep learning model, in the hypothesis of a real-time production app, you would have to factor in the resource consumption as well which would be way higher than other methods too. <br> Although the answer would also depends on the availibilty of GPU acceleration (costs more but way faster).

So the recommendation for this hypothesis can not be simple as yes or no, it would depend on the use-case :
- vader for real-time and cost-effective solutions, it can even analyze informal text
- for offline and batch precise analytics, maybe time and resource consumption are not a problem and flair would be the answer


3. The Next Step

- one could imagine an hybrid approach where easy cases (high VADER confidence) are VADER task only, and a small amount of ambiguous (low VADER confidence) cases go through the flair route.

this would counter the weaknesses of the two best models, VADER confusing ambiguous reviews (correct but not excellent accuracy), and flair time and resource heavy profile
